# InoveHub
## Sistema de Incubadora de Empresas

In [2]:
%pip install matplotlib seaborn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 1.1 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.3 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.3 MB 1.1 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.3 MB 1.1 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.3 MB 1.1 MB/s eta 0:00:06
   -------- ------------------------------- 1.8/8.3 MB 1.1 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.3 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.3 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.3 MB 1.1 MB/s eta 0:00:06
   ------------ --------------------------- 2.6/8.3 MB 1.1 MB/s eta 0:00:05
   ------------- ----------------

In [ ]:
import os
import pandas as pd
import panel as pn
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv

pn.extension('plotly') # Habilita o Plotly no Panel
load_dotenv()

@pn.cache
def load_data():
    DB_URL = os.getenv("DATABASE_URL")
    engine = create_engine(DB_URL)
    
    # Carregando tabelas
    df_emp = pd.read_sql("SELECT * FROM Empresa", engine)
    
    df_inv = pd.read_sql("SELECT * FROM Investimento", engine)
    df_inv['data_aporte'] = pd.to_datetime(df_inv['data_aporte'])
    
    df_cont = pd.read_sql("SELECT * FROM Contato", engine)
    
    return df_emp, df_inv, df_cont

# Carrega os dados uma vez
df_empresa, df_investimento, df_contato = load_data()

# Widget para filtrar Áreas de Atuação (Empresas)
areas_disponiveis = list(df_empresa['area_atuacao'].unique()) if not df_empresa.empty else []
filtro_area = pn.widgets.MultiChoice(
    name='Filtrar por Área de Atuação', 
    options=areas_disponiveis,
    value=areas_disponiveis, # Começa com todas selecionadas
    solid=False
)

# Widget para definir Top N cargos (Contatos)
slider_top_cargos = pn.widgets.IntSlider(
    name='Top N Cargos', start=3, end=20, step=1, value=10
)


def criar_grafico_funil_empresas(areas):
    # Filtra o DataFrame com base no widget
    if not areas: 
        df_filtrado = df_empresa # Se nada selecionado, mostra tudo
    else:
        df_filtrado = df_empresa[df_empresa['area_atuacao'].isin(areas)]
        
    if df_filtrado.empty:
        return pn.pane.Markdown("### Sem dados para exibir")

    # Contagem
    contagem = df_filtrado['status_atual'].value_counts().reset_index()
    contagem.columns = ['Status', 'Quantidade']
    
    # Plotly Bar Chart
    fig = px.bar(contagem, x='Status', y='Quantidade', color='Status',
                 title="Funil de Empresas na Incubadora (Status)", template="plotly_white")
    return fig

def criar_grafico_investimentos(areas_dummy): # Recebe areas só para atualizar junto, se quiser
    if df_investimento.empty: return pn.pane.Markdown("### Sem investimentos")
    
    # Agrupamento
    df_agrupado = df_investimento.groupby(df_investimento['data_aporte'].dt.to_period('M').astype(str))['valor'].sum().reset_index()
    
    fig = px.line(df_agrupado, x='data_aporte', y='valor', markers=True,
                  title="Evolução Financeira", template="plotly_white")
    fig.update_layout(xaxis_title="Mês/Ano", yaxis_title="Valor (R$)")
    return fig

def criar_grafico_contatos(top_n):
    if df_contato.empty: return pn.pane.Markdown("### Sem contatos")
    
    top_cargos = df_contato['cargo'].value_counts().nlargest(top_n).reset_index()
    top_cargos.columns = ['Cargo', 'Quantidade']
    
    fig = px.bar(top_cargos, y='Cargo', x='Quantidade', orientation='h',
                 title=f"Top {top_n} Cargos na Rede", color='Quantidade', template="plotly_white")
    fig.update_layout(yaxis={'categoryorder':'total ascending'}) # Ordena do maior pro menor
    return fig

grafico_empresa_view = pn.bind(criar_grafico_funil_empresas, areas=filtro_area)
grafico_invest_view = pn.bind(criar_grafico_investimentos, areas_dummy=filtro_area)
grafico_contato_view = pn.bind(criar_grafico_contatos, top_n=slider_top_cargos)

template = pn.template.FastListTemplate(
    title='InoveHub Dashboard',
    sidebar=[
        pn.pane.Markdown("## Filtros Gerais"),
        filtro_area,
        pn.layout.Divider(),
        pn.pane.Markdown("## Configuração Contatos"),
        slider_top_cargos,
        pn.layout.Divider(),
        pn.pane.Markdown("Dados carregados do PostgreSQL.")
    ],
    main=[
        pn.Row(
            pn.Column(grafico_empresa_view, margin=(10,10)),
            pn.Column(grafico_invest_view, margin=(10,10))
        ),
        pn.Row(
            pn.Column(grafico_contato_view, margin=(10,10))
        )
    ],
    accent_base_color="#2ecc71",
    header_background="#2c3e50"
)

# Comando para mostrar no notebook ou preparar para servir
template.servable();
template.show()

Launching server at http://localhost:63235
